# Comparación MOP–DP2 usando LSDB/HATS

Este notebook implementa una segunda ruta independiente para estudiar los matches: convierte los eventos MOP en un catálogo LSDB pequeño y lo cruza con el catálogo HATS de objetos DP2 disponible en el RSP.

La comparación es complementaria, no reemplaza la cobertura de visitas/coadds del notebook 01: LSDB cruza catálogos de puntos y devuelve objetos DP2 cercanos; no reconstruye directamente los footprints de detectores ni valida píxeles de coadds.

LSDB usa particiones HEALPix/HATS y catálogos marginados para evitar perder matches cerca de fronteras de partición. La documentación recomienda exigir un margen en el catálogo derecho para que el crossmatch sea completo.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import lsdb
from IPython.display import display
TARGET_SELECTION_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'target_selection' / 'cache_paths.py').exists()), None)
if TARGET_SELECTION_ROOT is None: raise RuntimeError('No se encontró la raíz del repositorio.')
if str(TARGET_SELECTION_ROOT) not in sys.path: sys.path.insert(0, str(TARGET_SELECTION_ROOT))
from target_selection.cache_paths import OUTPUT_ROOT
from target_selection.catalog_matching import crossmatch_lsdb
from target_selection.catalog_products import match_dp2_catalog, open_dp2_hats_catalog
OUT=OUTPUT_ROOT/'analyses'/'mop_dp2'/'lsdb_hats'
TABLES=OUTPUT_ROOT/'mop_dp2_all'/'dp2'/'tables'
LSDB_OUT=OUT/'lsdb_hats'
LSDB_OUT.mkdir(parents=True,exist_ok=True)
BANDS=tuple('ugrizy')
DP2_HATS_PATH='/rubin/lsdb_data/object_collection'
MATCH_RADIUS_ARCSEC=1.0
RUN_FULL_CROSSMATCH=False  # Reutiliza el CSV; cambiar a True sólo para recalcular los 6444 targets
SAMPLE_SIZE=200
print('LSDB:',lsdb.__version__)
print('DP2 HATS:',DP2_HATS_PATH)

## 1. Cargar la misma base MOP–DP2 usada en el estudio principal

Usamos `dp2_coadd_quality.csv` sólo como catálogo de posiciones y como referencia para comparar resultados. No volvemos a consultar TAP ni Butler.

In [ ]:
quality_path=TABLES/'dp2_coadd_quality.csv'
if not quality_path.exists(): raise FileNotFoundError(f'No existe {quality_path}')
catalog=pd.read_csv(quality_path,low_memory=False)
catalog=catalog.loc[:,~catalog.columns.astype(str).str.startswith('Unnamed:')].copy()
catalog['target_id']=catalog['target_id'].astype(str)
catalog['ra_deg']=pd.to_numeric(catalog['ra_deg'],errors='coerce')
catalog['dec_deg']=pd.to_numeric(catalog['dec_deg'],errors='coerce')
mop_points=catalog[['target_id','ra_deg','dec_deg']].dropna().drop_duplicates('target_id').copy()
mop_points['mop_row_id']=np.arange(len(mop_points),dtype='int64')
print(f'Targets MOP con coordenadas: {len(mop_points):,}')
display(mop_points.head())

## 2. Crear el catálogo HATS de MOP

`lsdb.from_dataframe` particiona los eventos por HEALPix. El margen de MOP no es el elemento crítico del crossmatch, porque MOP queda a la izquierda; el margen importante debe existir en el catálogo DP2 derecho.

In [ ]:
mop_lsdb = None  # La partición se construye dentro de crossmatch_lsdb.
print('Targets MOP disponibles para el matching:', len(mop_points))

## 3. Abrir DP2 HATS de forma perezosa

El path `/rubin/lsdb_data/object_collection` es el catálogo HATS de objetos DP2 documentado para el RSP. Sólo se solicitan coordenadas y un identificador; los datos se leen cuando se ejecuta `.compute()`. Si el catálogo no tiene margen cargado, la celda siguiente debe detenerse: hacer el crossmatch sin margen puede perder coincidencias en fronteras HEALPix.

In [ ]:
dp2=open_dp2_hats_catalog('object', columns=['coord_ra','coord_dec'], show_statistics=False)
print(dp2)
print('Margin DP2:',dp2.margin)
if dp2.margin is None: raise RuntimeError('El catálogo DP2 no tiene margin cache cargado; no continuar hasta configurar el margin correcto.')

## 4. Crossmatch de prueba

Se cruza MOP a la izquierda contra DP2 a la derecha. `n_neighbors=1` selecciona el objeto DP2 más cercano y `how='left'` conserva también los eventos MOP sin objeto asociado. La distancia queda en `_dist_arcsec`.

In [ ]:
trial_result=crossmatch_lsdb(
    mop_points.head(SAMPLE_SIZE), dp2, target_ra_column='ra_deg', target_dec_column='dec_deg',
    catalog_name='mop_trial', radius_arcsec=MATCH_RADIUS_ARCSEC, n_neighbors=1,
    require_right_margin=True, suffix_method='overlapping_columns', log_changes=False,
)
print('Filas del crossmatch de prueba:',len(trial_result))
print(trial_result.columns.tolist())
display(trial_result.head())

## 5. Crossmatch completo y cache local

La primera ejecución completa puede leer muchas particiones DP2, pero LSDB alinea los catálogos espacialmente y no necesita cargar toda la tabla DP2 en memoria como un DataFrame plano. El resultado materializado se guarda en CSV para que los análisis posteriores no repitan el crossmatch.

In [ ]:
result_path=LSDB_OUT/f'mop_dp2_object_crossmatch_{MATCH_RADIUS_ARCSEC:.2f}arcsec.csv'
if result_path.exists() and not RUN_FULL_CROSSMATCH:
    lsdb_result=pd.read_csv(result_path,low_memory=False)
    print('Reusing:',result_path)
else:
    lsdb_result=match_dp2_catalog(
        mop_points, product='object', output_path=result_path, reuse_existing=False,
        radius_arcsec=MATCH_RADIUS_ARCSEC, n_neighbors=1, require_right_margin=True,
        suffix_method='overlapping_columns', log_changes=False,
    )
    lsdb_result['lsdb_object_match'] = lsdb_result['object_match']
    print('Saved:',result_path)
print('Filas:',len(lsdb_result))
display(lsdb_result.head())

## 6. Resumir los matches LSDB

La columna `_dist_arcsec` permite definir el match. En un `left` crossmatch los no-match quedan con distancia nula/NaN. Revisar los nombres de columnas DP2 en la muestra antes de interpretar un identificador de objeto específico.

In [ ]:
distance_column='_dist_arcsec'
lsdb_result[distance_column]=pd.to_numeric(lsdb_result[distance_column],errors='coerce')
lsdb_result['lsdb_object_match']=lsdb_result[distance_column].le(MATCH_RADIUS_ARCSEC)
match_summary=pd.DataFrame([{'quantity':'mop_targets_in_input','value':len(mop_points)},{'quantity':'lsdb_rows','value':len(lsdb_result)},{'quantity':'lsdb_object_matches','value':int(lsdb_result['lsdb_object_match'].sum())},{'quantity':'lsdb_object_no_match','value':int((~lsdb_result['lsdb_object_match']).sum())},{'quantity':'match_radius_arcsec','value':MATCH_RADIUS_ARCSEC}])
display(match_summary)
lsdb_result.to_csv(LSDB_OUT/'mop_dp2_object_crossmatch_with_flags.csv',index=False)
match_summary.to_csv(LSDB_OUT/'mop_dp2_object_match_summary.csv',index=False)

## 7. Comparar LSDB con las banderas existentes

Esta comparación distingue cuatro conceptos: objeto DP2 cercano (LSDB), visita individual confirmada, coadd disponible y fotometría DIA medida. No esperamos que sean idénticos: un objeto DP2 puede no tener forced photometry útil, y una posición puede tener cobertura de coadd/visitas sin un objeto catalogado cercano.

In [ ]:
comparison=catalog.merge(lsdb_result[['target_id','lsdb_object_match',distance_column]],on='target_id',how='left') if 'target_id' in lsdb_result.columns else catalog.copy()
if 'lsdb_object_match' in comparison: comparison['lsdb_object_match']=comparison['lsdb_object_match'].fillna(False).astype(bool)
for c in ['has_visit_any','has_coadd','has_valid_coadd_any','dp2_dia_match']:
    if c in comparison: comparison[c]=comparison[c].fillna(False).astype(bool)
if 'lsdb_object_match' in comparison:
    comparison['lsdb_vs_existing']=np.select([comparison['lsdb_object_match'] & comparison.get('has_visit_any',False),comparison['lsdb_object_match'] & ~comparison.get('has_visit_any',False),~comparison['lsdb_object_match'] & comparison.get('has_visit_any',False)],['lsdb_and_visit','lsdb_only','visit_only'],default='neither')
    display(comparison['lsdb_vs_existing'].value_counts().rename_axis('category').to_frame('targets'))
comparison.to_csv(LSDB_OUT/'mop_dp2_lsdb_comparison.csv',index=False)

## Interpretación y límites

* Este notebook valida un match de objetos DP2 mediante LSDB/HATS, no reemplaza la consulta de `VisitDetector` ni la consulta Butler de coadds.
* El resultado debe compararse por target y por distancia, no sólo por el número total.
* La comparación más importante será LSDB object match vs `dp2_dia_match`, `has_visit_any` y `has_valid_coadd_any`.
* Si se necesita una prueba de cobertura pura con LSDB, hay que disponer de un catálogo HATS de footprints o de mapas DP2; el `object_collection` no representa por sí mismo toda la cobertura de imágenes.
* La primera ejecución recomendada es una muestra de 200 targets. Sólo después de revisar columnas, margen, radio y memoria se activa `RUN_FULL_CROSSMATCH=True`.

## 8. Crossmatch independiente contra `DiaObject`

La rama anterior cruzó MOP contra `Object`, es decir, fuentes detectadas en coadds. Esta rama reutiliza el mismo catálogo MOP y el mismo catálogo LSDB particionado, pero consulta `dia_object_collection`.

`dia_object_collection` contiene el catálogo DP2 `DiaObject` y, en columnas anidadas, `diaSource` y `diaObjectForcedSource` (`ForcedSourceOnDiaObject`). Por eso es la comparación LSDB más cercana al pipeline original:

```text
MOP → DiaObject cercano → diaObjectForcedSource
```

Para el crossmatch sólo cargamos coordenadas, `diaObjectId` y `nDiaSources`; las curvas de luz anidadas se mantienen para una etapa posterior y no se cargan durante esta corrida.

In [ ]:
DIA_OBJECT_HATS_PATH = '/rubin/lsdb_data/dia_object_collection'
DIA_RUN_FULL_CROSSMATCH = False  # Cambiar a True sólo para recalcular el crossmatch DiaObject.

dia_dp2 = open_dp2_hats_catalog(
    'dia_object',
    columns=['ra', 'dec', 'diaObjectId', 'nDiaSources'],
    show_statistics=False,
)
print('Catálogo HATS DiaObject:', dia_dp2)
print('Margin DiaObject:', dia_dp2.margin)
if dia_dp2.margin is None:
    raise RuntimeError('dia_object_collection no tiene margin cache cargado; no continuar.')

### 8.1 Crossmatch completo MOP–DiaObject

Se mantiene `how='left'` para conservar los 6.444 targets MOP, incluidos los que no tienen `DiaObject`. El radio es el mismo de la comparación con `Object` y el margen del catálogo derecho se exige para no perder matches en los bordes de las particiones HEALPix.

In [ ]:
dia_result_path = LSDB_OUT / f'mop_dp2_diaobject_crossmatch_{MATCH_RADIUS_ARCSEC:.2f}arcsec.csv'
if dia_result_path.exists() and not DIA_RUN_FULL_CROSSMATCH:
    dia_result = pd.read_csv(dia_result_path, low_memory=False)
    print('Reutilizando:', dia_result_path)
else:
    dia_result = match_dp2_catalog(
        mop_points, product='dia_object', output_path=dia_result_path, reuse_existing=False,
        radius_arcsec=MATCH_RADIUS_ARCSEC, n_neighbors=1, require_right_margin=True,
        suffix_method='overlapping_columns', log_changes=False,
    )
    dia_result['lsdb_diaobject_match'] = dia_result['dia_object_match']
    print('Saved:', dia_result_path)

dia_result['_dist_arcsec'] = pd.to_numeric(dia_result['_dist_arcsec'], errors='coerce')
dia_result['lsdb_diaobject_match'] = dia_result['_dist_arcsec'].le(MATCH_RADIUS_ARCSEC)
dia_result.to_csv(LSDB_OUT / 'mop_dp2_diaobject_crossmatch_with_flags.csv', index=False)

dia_summary = pd.DataFrame([
    {'quantity': 'mop_targets_in_input', 'value': len(mop_points)},
    {'quantity': 'diaobject_crossmatch_rows', 'value': len(dia_result)},
    {'quantity': 'lsdb_diaobject_matches', 'value': int(dia_result['lsdb_diaobject_match'].sum())},
    {'quantity': 'lsdb_diaobject_no_match', 'value': int((~dia_result['lsdb_diaobject_match']).sum())},
    {'quantity': 'match_radius_arcsec', 'value': MATCH_RADIUS_ARCSEC},
])
display(dia_summary)
dia_summary.to_csv(LSDB_OUT / 'mop_dp2_diaobject_match_summary.csv', index=False)
display(dia_result.loc[dia_result['lsdb_diaobject_match']].head())

### 8.2 Comparación conjunta: `Object`, `DiaObject` y cobertura

Esta tabla no interpreta la ausencia de un catálogo como ausencia de datos de imagen. Separa explícitamente:

- match con `Object`;
- match con `DiaObject`;
- visita registrada;
- coadd válido.

El resultado permite identificar eventos que sólo aparecen como variables (`DiaObject`), sólo como fuentes de coadd (`Object`), aparecen en ambos, o tienen cobertura de imagen sin una detección catalogada.

In [ ]:
object_flags = lsdb_result[['target_id', 'lsdb_object_match', '_dist_arcsec']].rename(
    columns={'_dist_arcsec': 'object_match_distance_arcsec'}
)
dia_flags = dia_result[['target_id', 'lsdb_diaobject_match', '_dist_arcsec', 'diaObjectId', 'nDiaSources']].rename(
    columns={'_dist_arcsec': 'diaobject_match_distance_arcsec'}
)

joint = (catalog
         .merge(object_flags, on='target_id', how='left')
         .merge(dia_flags, on='target_id', how='left'))
if 'has_visit_any' not in joint and 'has_valid_visit_any' in joint:
    joint['has_visit_any'] = joint['has_valid_visit_any'].fillna(False).astype(bool)
for col in ['lsdb_object_match', 'lsdb_diaobject_match', 'has_visit_any', 'has_valid_coadd_any']:
    if col in joint:
        joint[col] = joint[col].fillna(False).astype(bool)

joint['catalog_match_category'] = np.select(
    [
        joint['lsdb_object_match'] & joint['lsdb_diaobject_match'],
        joint['lsdb_object_match'] & ~joint['lsdb_diaobject_match'],
        ~joint['lsdb_object_match'] & joint['lsdb_diaobject_match'],
        joint['has_visit_any'],
    ],
    ['object_and_diaobject', 'object_only', 'diaobject_only', 'coverage_without_catalog_match'],
    default='no_catalog_match_and_no_visit',
)

joint.to_csv(LSDB_OUT / 'mop_dp2_lsdb_object_vs_diaobject_comparison.csv', index=False)
joint_summary = joint['catalog_match_category'].value_counts().rename_axis('category').to_frame('targets')
display(joint_summary)
joint_summary.to_csv(LSDB_OUT / 'mop_dp2_lsdb_object_vs_diaobject_summary.csv')

print('Targets:', len(joint))
print('Object matches:', int(joint['lsdb_object_match'].sum()))
print('DiaObject matches:', int(joint['lsdb_diaobject_match'].sum()))
print('Matches en ambos:', int((joint['lsdb_object_match'] & joint['lsdb_diaobject_match']).sum()))

## 9. Interpretación y acceso posterior a las curvas de luz DiaObject

El crossmatch anterior sólo consulta las columnas de identificación. El catálogo HATS completo también contiene las columnas anidadas `diaSource` y `diaObjectForcedSource`. Para un target con `diaObjectId`, esas columnas permiten extraer las detecciones diferenciales y la fotometría forced sin repetir una consulta TAP por target.

La advertencia importante es que un `DiaObject` requiere asociación de `DiaSource`; por lo tanto, un target puede tener visitas, coadds o incluso fotometría de imagen forced sin aparecer en `dia_object_collection`. La ausencia de `DiaObject` no debe interpretarse como ausencia de cobertura DP2.

## 10. Visualizaciones de intersecciones y categorías

Estas celdas no vuelven a consultar LSDB. Usan `joint`, la tabla conjunta guardada en la sección anterior. Si se abre el notebook directamente desde esta sección, se puede cargar el CSV guardado.

In [ ]:
from target_selection.plotting import (
    plot_upset, plot_match_skyplot, plot_upset_with_band_heatmap,
    plot_upset_with_band_bars, plot_upset_with_band_conditions, save_figure,
)

plots_dir = LSDB_OUT / 'plots'
joint_path = LSDB_OUT / 'mop_dp2_lsdb_object_vs_diaobject_comparison.csv'
if 'joint' not in globals():
    joint = pd.read_csv(joint_path, low_memory=False)
if 'has_visit_any' not in joint and 'has_valid_visit_any' in joint:
    joint['has_visit_any'] = joint['has_valid_visit_any'].fillna(False).astype(bool)
print('Targets para visualizaciones:', len(joint))

### 10.1 UpSet de cobertura e imágenes

Cada columna representa una combinación de cualidades. La barra superior es el número de targets en esa combinación; los puntos inferiores muestran qué cualidades están presentes.

In [ ]:
coverage_columns = [
    'has_visit_any',
    'has_coadd',
    'has_valid_metrics_any',
    'has_valid_pixels_any',
    'has_valid_coadd_any',
]
coverage_labels = ['alguna visita registrada', 'coadd', 'métricas válidas en alguna banda', 'píxeles válidos en alguna banda', 'coadd completo en alguna banda']
fig_coverage, coverage_intersections = plot_upset(
    joint,
    coverage_columns,
    labels=coverage_labels,
    title='Intersecciones: cobertura y calidad de imágenes',
    max_intersections=20,
)
coverage_intersections.to_csv(LSDB_OUT / 'coverage_intersections.csv', index=False)
save_figure(fig_coverage, plots_dir / 'upset_coverage.png')

### 10.2 UpSet de catálogos y productos de curvas de luz

Esta figura muestra qué targets tienen un `Object`, un `DiaObject`, cobertura válida y, si ya fue calculado, fotometría DIA/forced.

In [ ]:
catalog_columns = [
    'lsdb_object_match',
    'lsdb_diaobject_match',
    'has_visit_any',
    'has_valid_coadd_any',
]
catalog_labels = ['Object LSDB', 'DiaObject LSDB', 'alguna visita registrada', 'coadd completo en alguna banda']
fig_catalogs, catalog_intersections = plot_upset(
    joint,
    catalog_columns,
    labels=catalog_labels,
    title='Intersecciones: catálogos LSDB y cobertura DP2',
    max_intersections=20,
)
catalog_intersections.to_csv(LSDB_OUT / 'catalog_intersections.csv', index=False)
save_figure(fig_catalogs, plots_dir / 'upset_catalogs.png')

### 10.3 Skyplot por categoría

Cada target aparece una vez. El color permite ver dónde están los targets sin visita registrada, los que tienen cobertura pero no match catalogado, los que tienen sólo `Object`, sólo `DiaObject` o ambos.

In [ ]:
fig_sky = plot_match_skyplot(joint)
save_figure(fig_sky, plots_dir / 'skyplot_match_categories.png')

### 10.4 UpSet completo ordenado por completitud

Esta versión incluye simultáneamente las cualidades de cobertura, calidad de imagen y catálogos LSDB. No ordena por cantidad de targets: empieza por la combinación que cumple todas las condiciones y luego muestra combinaciones que van perdiendo una, dos, tres, etc. cualidades.

In [ ]:
# Normaliza el nombre de la condición de cobertura antes de construir el UpSet.
# Algunas tablas antiguas guardan esta columna como has_valid_visit_any.
if 'has_visit_any' not in joint and 'has_valid_visit_any' in joint:
    joint = joint.copy()
    joint['has_visit_any'] = joint['has_valid_visit_any'].fillna(False).astype(bool)

full_columns = [
    'has_visit_any',
    'has_coadd',
    'has_valid_metrics_any',
    'has_valid_pixels_any',
    'has_valid_coadd_any',
    'lsdb_object_match',
    'lsdb_diaobject_match',
]
full_labels = [
    'alguna visita registrada',
    'coadd',
    'métricas válidas en alguna banda',
    'píxeles válidos en alguna banda',
    'coadd completo en alguna banda',
    'Object LSDB',
    'DiaObject LSDB',
]
fig_full, full_intersections = plot_upset(
    joint,
    full_columns,
    labels=full_labels,
    title='Cobertura DP2 y matches de catálogos',
    max_intersections=None,
    order='completeness',
)

full_intersections.to_csv(LSDB_OUT / 'full_quality_intersections_by_completeness.csv', index=False)
save_figure(fig_full, plots_dir / 'upset_full_quality_by_completeness.png')

### 10.5 UpSet con bandas, sin duplicar targets

La barra sigue contando cada target una sola vez por combinación. La matriz de colores inferior muestra, para cada combinación, qué fracción de sus targets cumple la propiedad elegida en cada banda. En esta primera versión se muestra `coadd completo` por banda.

In [ ]:
band_property_columns = {band: f'has_valid_coadd_{band}' for band in BANDS}
fig_band, band_intersections, band_fractions = plot_upset_with_band_heatmap(
    joint,
    intersection_columns=full_columns,
    band_property_columns=band_property_columns,
    labels=full_labels,
    title='Cobertura y matches: coadd completo por banda',
    order='completeness',
)
band_intersections.to_csv(LSDB_OUT / 'full_quality_intersections_with_band_coadds.csv', index=False)
pd.DataFrame(band_fractions, index=list(BANDS)).to_csv(LSDB_OUT / 'full_quality_band_fractions.csv')
save_figure(fig_band, plots_dir / 'upset_full_quality_with_band_coadds.png')

### 10.6 Barras por banda dentro de cada combinación

Cada grupo corresponde a una combinación del UpSet y contiene una barra para cada banda. La altura es la fracción de los targets de esa combinación que cumplen, en esa misma banda, todas las condiciones positivas de la combinación. Un target puede contribuir en varias bandas; por eso las barras no se suman al total.

In [ ]:
# Normaliza el alias de cobertura por si el notebook se ejecutó desde una
# tabla que conserva el nombre técnico has_valid_visit_any.
if 'has_visit_any' not in joint and 'has_valid_visit_any' in joint:
    joint = joint.copy()
    joint['has_visit_any'] = joint['has_valid_visit_any'].fillna(False).astype(bool)

required_intersection_columns = [
    'has_visit_any', 'has_coadd', 'has_valid_metrics_any',
    'has_valid_pixels_any', 'has_valid_coadd_any',
    'lsdb_object_match', 'lsdb_diaobject_match',
]
missing_intersection_columns = [c for c in required_intersection_columns if c not in joint]
if missing_intersection_columns:
    raise KeyError(f'Faltan columnas para el UpSet: {missing_intersection_columns}')

missing_band_columns = [
    f'{prefix}_{band}'
    for prefix in ('has_valid_visit', 'has_coadd', 'has_valid_metrics', 'has_valid_pixels', 'has_valid_coadd')
    for band in BANDS
    if f'{prefix}_{band}' not in joint
]
if missing_band_columns:
    raise KeyError(f'Faltan columnas por banda para el gráfico: {missing_band_columns}')

# Relaciona cada condición general del UpSet con su columna por banda.
# Object/DiaObject no aparecen aquí porque no dependen de la banda.
band_condition_columns = {
    'has_visit_any': {band: f'has_valid_visit_{band}' for band in BANDS},
    'has_coadd': {band: f'has_coadd_{band}' for band in BANDS},
    'has_valid_metrics_any': {band: f'has_valid_metrics_{band}' for band in BANDS},
    'has_valid_pixels_any': {band: f'has_valid_pixels_{band}' for band in BANDS},
    'has_valid_coadd_any': {band: f'has_valid_coadd_{band}' for band in BANDS},
}
fig_band_bars, band_bar_intersections, band_bar_counts, band_bar_fractions = plot_upset_with_band_bars(
    joint,
    intersection_columns=full_columns,
    band_condition_columns=band_condition_columns,
    labels=full_labels,
    title='Cobertura y matches: condiciones cumplidas por banda dentro de cada combinación',
    order='completeness',
)
display(fig_band_bars)
band_bar_intersections.to_csv(LSDB_OUT / 'full_quality_intersections_with_band_bars.csv', index=False)
pd.DataFrame(
    band_bar_counts, index=list(BANDS),
).to_csv(LSDB_OUT / 'full_quality_band_bar_counts.csv')
pd.DataFrame(
    band_bar_fractions, index=list(BANDS),
).to_csv(LSDB_OUT / 'full_quality_band_bar_fractions.csv')
save_figure(fig_band_bars, plots_dir / 'upset_full_quality_with_band_bars.png')

### 10.7 Condiciones independientes por banda

This plot does not require all conditions to be present in the same band. For each combination, it counts targets with visits, coadds, valid metrics, and finite pixels independently in each band. This makes cross-band cases visible.

In [ ]:
condition_band_columns = {
    'visit': {band: f'has_valid_visit_{band}' for band in BANDS},
    'coadd': {band: f'has_coadd_{band}' for band in BANDS},
    'valid_metrics': {band: f'has_valid_metrics_{band}' for band in BANDS},
    'finite_pixels': {band: f'has_valid_pixels_{band}' for band in BANDS},
}
full_labels_en = [
    'Any visit', 'Any coadd', 'Valid metrics', 'Finite pixels',
    'Complete coadd', 'Object LSDB', 'DiaObject LSDB',
]
fig_condition_band, condition_band_intersections, condition_band_counts, condition_band_fractions, condition_band_rows = plot_upset_with_band_conditions(
    joint,
    intersection_columns=full_columns,
    condition_band_columns=condition_band_columns,
    labels=full_labels_en,
    title='Coverage and catalog matches: conditions by band',
    order='completeness',
)

condition_band_intersections.to_csv(LSDB_OUT / 'full_quality_intersections_with_band_conditions.csv', index=False)
pd.DataFrame(condition_band_counts, index=pd.MultiIndex.from_tuples(condition_band_rows, names=['condition', 'band'])).to_csv(LSDB_OUT / 'full_quality_band_condition_counts.csv')
pd.DataFrame(condition_band_fractions, index=pd.MultiIndex.from_tuples(condition_band_rows, names=['condition', 'band'])).to_csv(LSDB_OUT / 'full_quality_band_condition_fractions.csv')
save_figure(fig_condition_band, plots_dir / 'upset_full_quality_with_band_conditions.png')